In [1]:
import re
import os
from pathlib import Path
import json
import logging

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rich import print, pretty, inspect
from rich.console import Console

In [2]:
console = Console()
console.print("hello", style = "bold white")

hello

In [3]:
def init_logger(log_level):
    """Initialize a logger instance"""
    logger = logging.getLogger(__name__)
    logger.setLevel(log_level)
    console_handler = logging.StreamHandler()
    console_handler.setLevel(log_level)
    FORMAT = "[%(levelname)s][%(asctime)s][%(filename)s:%(lineno)s - %(funcName)10s() ] %(message)s"
    formatter = logging.Formatter(FORMAT)
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    return logger

lg = init_logger(logging.INFO)

### 0. Extract All Lines Modifying ```this_block```
Example wrapper file used for dev: [casper_wb_fft_config.m](https://github.com/talonmyburgh/casper_dspdevel/blob/a8b6f3d9311d7f9579a6eb7015d4d549fece36fe/wrappers/simulink/casper_wb_fft_config.m)

Wei Liu's example SciLab port for ```casper_wb_fft_config.m```:
1. JSON: [wbfft.json](https://github.com/liuweiseu/mlib_devel/blob/c74c7bae6b34bd3d8231a78a75955ab2a48cf24e/scilab_library/scilab_blocks/casper_dsp/wbfft.json)
2. SciLab: [wbfft.sci](https://github.com/liuweiseu/mlib_devel/blob/c74c7bae6b34bd3d8231a78a75955ab2a48cf24e/scilab_library/scilab_blocks/casper_dsp/wbfft.sci)
3. Python: [wbfft.py](https://github.com/liuweiseu/mlib_devel/blob/c74c7bae6b34bd3d8231a78a75955ab2a48cf24e/scilab_library/dsp_blocks/wbfft.py)

In [4]:
wrappers_root = Path("simulink")
scilab_translations_root = Path("scilab")
os.makedirs(scilab_translations_root, exist_ok=True)

scilab_dsp_root = "casper_dspdevel"

def match_target_lines(target_fpath, line_pat):
    with open(target_fpath, "r") as f:
        d = f.read()
        matches = re.findall(line_pat, d)
        match_line_df = pd.DataFrame(matches)
    return match_line_df

def match_this_block(target_file, this_block_pat="this_block\.(.*)\((.*)\);"):
    this_block_lines_df = match_target_lines(
        target_file,
        line_pat=this_block_pat
    )
    return this_block_lines_df.rename(columns={0: "fn", 1: "args"})

In [5]:
target_file = "edge_detect_config.m"
target_file = "casper_wb_fft_config.m"

target_fpath_test = wrappers_root / target_file

In [6]:
# Demonstration: 30 random lines containing the pattern "this_block.[fn](args)"
match_df = match_this_block(target_fpath_test)
match_df.sample(min(len(match_df), 30))

,fn,args
44,addSimulinkOutport,out_im_port
27,port,'out_sync'
76,addFileToLibrary,[filepath '/../../common_components/common_asy...
123,addFileToLibrary,[filepath '/../../casper_wb_fft/fft_r2_wide.vh...
24,addSimulinkInport,in_re_port
10,addSimulinkInport,'in_bsn'
38,addSimulinkOutport,'out_empty'
75,addFileToLibrary,[filepath '/../../casper_adder/common_add_sub....
122,addFileToLibrary,[filepath '/../../casper_wb_fft/fft_sepa_wide....
39,port,'out_empty'


### 1. Extract File dependencies

Regex patterns ([wbfft example](https://github.com/talonmyburgh/casper_dspdevel/blob/a8b6f3d9311d7f9579a6eb7015d4d549fece36fe/wrappers/simulink/casper_wb_fft_config.m#L246))

1. Find [import lines](https://regex101.com/r/gilGkt/1): ```this_block.addFileToLibrary(...);```
2. Match and extract file paths: 

In [7]:
## Extract file dependencies

def get_file_dependencies(target_fpath):
    # Regular expressions
    dep_fpaths_pat = "\[filepath '(.*)'\]"
    relative_path_pat = "[/\.]*(.*)"
    
    # Load relevant lines
    df = match_this_block(target_fpath)
    
    # Filter by function
    fns = ["addFileToLibrary"]
    df = df[df['fn'].isin(fns)]
    
    # Extract arguments + rename
    args_df = df['args'].str.split(',', expand=True)
    args_df = args_df.rename(columns={0: "simulink_fpath", 1: "lib"})
    df = pd.concat([args_df, df], axis=1)
    
    # Extract simulink fpaths and translate to scilab fpaths
    df['simulink_fpath'] = df['simulink_fpath'].str.extract(dep_fpaths_pat)
    df['scilab_fpath'] = scilab_dsp_root + "/" + (df['simulink_fpath'].str.extract(relative_path_pat))
    
    # Drop any imports that don't have first argument starting with "filename"
    df = df.dropna().reset_index(drop=True)
    return df

dep_df = get_file_dependencies(target_fpath_test)
dep_df.head(5)

,simulink_fpath,lib,fn,args,scilab_fpath
0,/../../common_pkg/fixed_float_types_c.vhd,'common_pkg_lib',addFileToLibrary,[filepath '/../../common_pkg/fixed_float_types...,casper_dspdevel/common_pkg/fixed_float_types_c...
1,/../../common_pkg/fixed_pkg_c.vhd,'common_pkg_lib',addFileToLibrary,[filepath '/../../common_pkg/fixed_pkg_c.vhd']...,casper_dspdevel/common_pkg/fixed_pkg_c.vhd
2,/../../common_pkg/common_pkg.vhd,'common_pkg_lib',addFileToLibrary,"[filepath '/../../common_pkg/common_pkg.vhd'],...",casper_dspdevel/common_pkg/common_pkg.vhd
3,/../../common_components/common_pipeline.vhd,'common_components_lib',addFileToLibrary,[filepath '/../../common_components/common_pip...,casper_dspdevel/common_components/common_pipel...
4,/../../casper_adder/common_add_sub.vhd,'casper_adder_lib',addFileToLibrary,[filepath '/../../casper_adder/common_add_sub....,casper_dspdevel/casper_adder/common_add_sub.vhd


### 2. Extract Generic Parameters
[Comment on supported generics](https://github.com/talonmyburgh/casper_dspdevel/blob/a8b6f3d9311d7f9579a6eb7015d4d549fece36fe/wrappers/simulink/casper_wb_fft_config.m#L222):
> %      The addGeneric function takes  3 parameters, generic name, type and constant value.
> Supported types are boolean, real, integer and string.

Regex patterns

1. Find lines with generics.
2. Match and extract file paths: 

In [8]:
## Extract generics

def get_generic_params(target_fpath):
    # Load "subsystem mask parameters for dynamic ports" (i.e. parent-controlled generics)
    parent_params_df = match_target_lines(
        target_fpath,
        line_pat = "get_param\((.*),(.*?)\)+"
    )
    parent_params_df = parent_params_df.rename(columns={0: 'parent_blk_name', 1: 'parent_generic_name'})
    # parent_params_df['parent_generic_name'] = parent_params_df['parent_generic_name'].str.replace("'", "")
    
    # Load this_block params
    this_block_params_df = match_target_lines(
        target_fpath,
        line_pat = "addGeneric\((.*),(.*),(.*)\)"
    )
    
    this_block_params_df = this_block_params_df.rename(columns={0: 'internal_generic_name', 1: 'type', 2: 'constant_value'})
    # this_block_params_df['internal_generic_name'] = this_block_params_df['internal_generic_name'].str.replace("'", "")

    # Combine parameters into one DataFrame
    df = pd.concat([parent_params_df, this_block_params_df])
    df['internal_generic_name']
    return df

gen_df = get_generic_params(target_fpath_test)
gen_df

,parent_blk_name,parent_generic_name,internal_generic_name,type,constant_value
0,wb_fft_blk,'Parent',NaN,NaN,NaN
1,wb_fft_blk_parent,'use_reorder',NaN,NaN,NaN
2,wb_fft_blk_parent,'use_fft_shift',NaN,NaN,NaN
3,wb_fft_blk_parent,'use_separate',NaN,NaN,NaN
4,wb_fft_blk_parent,'alt_output',NaN,NaN,NaN
5,wb_fft_blk_parent,'wb_factor',NaN,NaN,NaN
6,wb_fft_blk_parent,'nof_points',NaN,NaN,NaN
7,wb_fft_blk_parent,'in_dat_w',NaN,NaN,NaN
8,wb_fft_blk_parent,'out_dat_w',NaN,NaN,NaN
9,wb_fft_blk_parent,'out_gain_w',NaN,NaN,NaN


### 3. Extract I/O Ports

Example lines from wbfft:
- [static ports](https://github.com/talonmyburgh/casper_dspdevel/blob/a8b6f3d9311d7f9579a6eb7015d4d549fece36fe/wrappers/simulink/casper_wb_fft_config.m#L100)
- [dynamic ports](https://github.com/talonmyburgh/casper_dspdevel/blob/a8b6f3d9311d7f9579a6eb7015d4d549fece36fe/wrappers/simulink/casper_wb_fft_config.m#L147)

In [9]:
## Extract I/O Ports

# Regular expressions

def get_io_ports(target_fpath):
    # Load relevant lines
    df = match_this_block(target_fpath)
    
    # Filter by function
    fns = ["port", "addSimulinkInport", "addSimulinkOutport"]
    df = df[df['fn'].isin(fns)]
    
    # Extract arguments + rename
    args_df = df['args'].str.split(',', expand=True)
    args_df = args_df.rename(columns={0: "port_name"})
    df = pd.concat([args_df, df], axis=1)
    
    # Extract simulink fpaths and translate to scilab fpaths
    
    # Drop any imports that don't have first argument starting with "filename"
    df = df.dropna().reset_index(drop=True)
    return df

port_df = get_io_ports(target_fpath_test)
port_df.head(20)

,port_name,fn,args
0,'rst',addSimulinkInport,'rst'
1,'rst',port,'rst'
2,'in_sync',addSimulinkInport,'in_sync'
3,'in_sync',port,'in_sync'
4,'in_valid',addSimulinkInport,'in_valid'
5,'in_valid',port,'in_valid'
6,'in_shiftreg',addSimulinkInport,'in_shiftreg'
7,'in_shiftreg',port,'in_shiftreg'
8,'in_bsn',addSimulinkInport,'in_bsn'
9,'in_bsn',port,'in_bsn'


### 5. Build JSON config


In [10]:
def get_module_name(target_fpath):
    # Load relevant lines
    df = match_this_block(target_fpath)
    fns = ["setEntityName"]
    df = df[df['fn'].isin(fns)]

    name = ""
    if len(df) == 1:
        name_arg = df.iloc[0, 1]
        if isinstance(name_arg, str):
            name = name_arg.strip().replace("'", "")
        else:
            lg.error("Extracted name is not a string.")
    elif len(df) > 1:
        lg.error("No module name detected")
    else:
        lg.error("Multiple module names detected")

    return name



def init_config(target_fpath, tag_prefix="dsp"):
    # Fetch module name
    name = get_module_name(target_fpath_test)
    if name == "":
        lg.error("No name detected: cannot build config")
        return -1

    # Build minimal config
    json_config = {
        "parameters": {
            "keys": [
                "name",
                "fullpath",
                "tag",
            ],
            "values": [
                name,
                "",
                f"{tag_prefix}:{name}",
                
            ]
        }
    }

    # Add generic parameters
    
    default_param_value_template = "UNKNOWN-DEFAULT_{0}" 
    def add_gen_param(param):
        """Note: The config file will explicitly mark unknown defaults.
        A casper expert should fill in these defaults."""
        default_val = default_param_value_template.format(param)
        
        json_config['parameters']['keys'].append(param)
        json_config['parameters']['values'].append(default_val)

    # Add generic parameters
    gen_df = get_generic_params(target_fpath)
    parent_params_df = gen_df.loc[:, ["parent_blk_name", "parent_generic_name"]].dropna()
    parent_params_df["parent_generic_name"] = parent_params_df["parent_generic_name"].str.replace("'", "").str.strip()
    parent_params_df = parent_params_df[parent_params_df['parent_generic_name'] != 'Parent']
    for param in parent_params_df['parent_generic_name']:
        add_gen_param(param)
    
    return json_config

jc = init_config(target_fpath_test)
print(jc)

{
    'parameters': {
        'keys': [
            'name',
            'fullpath',
            'tag',
            'use_reorder',
            'use_fft_shift',
            'use_separate',
            'alt_output',
            'wb_factor',
            'nof_points',
            'in_dat_w',
            'out_dat_w',
            'out_gain_w',
            'stage_dat_w',
            'twid_dat_w',
            'max_addr_w',
            'guard_w',
            'guard_enable',
            'use_variant',
            'vendor_technology',
            'pipe_reo_in_place',
            'use_dsp',
            'ovflw_behav',
            'use_round',
            'ram_primitive',
            'xtra_dat_sigs'
        ],
        'values': [
            'wideband_fft_top',
            '',
            'dsp:wideband_fft_top',
            'UNKNOWN-DEFAULT_use_reorder',
            'UNKNOWN-DEFAULT_use_fft_shift',
            'UNKNOWN-DEFAULT_use_separate',
            'UNKNOWN-DEFAULT_alt_output',
            'UNKNOWN-DEFAULT_wb_factor',
            'UNKNOWN-DEFAULT_nof_points',
            'UNKNOWN-DEFAULT_in_dat_w',
            'UNKNOWN-DEFAULT_out_dat_w',
            'UNKNOWN-DEFAULT_out_gain_w',
            'UNKNOWN-DEFAULT_stage_dat_w',
            'UNKNOWN-DEFAULT_twid_dat_w',
            'UNKNOWN-DEFAULT_max_addr_w',
            'UNKNOWN-DEFAULT_guard_w',
            'UNKNOWN-DEFAULT_guard_enable',
            'UNKNOWN-DEFAULT_use_variant',
            'UNKNOWN-DEFAULT_vendor_technology',
            'UNKNOWN-DEFAULT_pipe_reo_in_place',
            'UNKNOWN-DEFAULT_use_dsp',
            'UNKNOWN-DEFAULT_ovflw_behav',
            'UNKNOWN-DEFAULT_use_round',
            'UNKNOWN-DEFAULT_ram_primitive',
            'UNKNOWN-DEFAULT_xtra_dat_sigs'
        ]
    }
}

### 6. Build Python DSP class

The code block below will output a module with the following format:
```python
class HDLProcessor(BaseProcessor):
    def initialize(self):
        'Initialize method'
        pass
    def modify_top(self):
        'Modify Top method'
        pass
    def _generate_hdl_wrapper(self):
        'Generate Hdl Wrapper method'
        pass

```

In [11]:
"""
DSP Block generator 
"""

import ast

def create_defaultdict_ast(dict_name: str) -> str:
    """Generates AST code to initialize a defaultdict(list) with a given name."""
    
    # Create import statement: from collections import defaultdict
    import_node = ast.ImportFrom(
        module='collections',
        names=[ast.alias(name='defaultdict', asname=None)],
        level=0
    )

    # Create assignment: {dict_name} = defaultdict(list)
    assign_node = ast.Assign(
        targets=[ast.Name(id=dict_name, ctx=ast.Store())],
        value=ast.Call(
            func=ast.Name(id='defaultdict', ctx=ast.Load()),
            args=[ast.Name(id='list', ctx=ast.Load())],
            keywords=[]
        )
    )

    return [import_node, assign_node]


def generate_default_method(method_name):
    """Empty method body (just pass)"""
    method_body = [
        ast.Expr(ast.Constant(value=f"{method_name.replace('_', ' ').title()} method")),
        ast.Pass()
    ]
    
    method = ast.FunctionDef(
        name=method_name,
        args=ast.arguments(
            posonlyargs=[],
            args=[ast.arg(arg='self')],
            kwonlyargs=[],
            kw_defaults=[],
            defaults=[]
        ),
        body=method_body,
        decorator_list=[],
        lineno=1,  # Add default lineno
        end_lineno=1  # Add end_lineno for Python 3.9+
    )
    return method

def generate_class(class_methods: dict, class_name: str, parent_class: str = "DSPBlock") -> str:
    """Generates a Python class with specified structure using AST nodes"""
    # print(class_methods)
    
    methods = []
    for method_name in ['initialize', 'modify_top', '_generate_hdl_wrapper']:
        if method_name in class_methods and isinstance(class_methods[method_name], ast.FunctionDef):
            method = class_methods[method_name]
        else:
            method = generate_default_method(method_name)
        methods.append(method)

    bases = [ast.Name(id=parent_class, ctx=ast.Load())] if parent_class else []

    class_def = ast.ClassDef(
        name=class_name,
        bases=bases,
        keywords=[],
        body=methods,
        decorator_list=[],
        lineno=1,  # Add lineno for class definition
        end_lineno=1
    )

    module = ast.Module(
        body=[
            *create_defaultdict_ast(class_name), # create import library defaultdict with list construct
            class_def
        ],
        type_ignores=[]
    )
    ast.fix_missing_locations(module)  # Critical fix for location attributes
    return ast.unparse(module)

# Example usage:
generated_code = generate_class(
    class_methods={},
    class_name="HDLProcessor"
)
print(generated_code)


from collections import defaultdict
HDLProcessor = defaultdict(list)

class HDLProcessor(DSPBlock):

    def initialize(self):
        """Initialize method"""
        pass

    def modify_top(self):
        """Modify Top method"""
        pass

    def _generate_hdl_wrapper(self):
        """ Generate Hdl Wrapper method"""
        pass

In [12]:
"""create code for the initialize method"""

def generate_library_dict_append(class_name, lib_name, path_name):
    # Build the subscript expression: <class_name>_libs[lib_name]
    lib_dict_name = f"{class_name}_libs"
    subscript = ast.Subscript(
        value=ast.Name(id=lib_dict_name, ctx=ast.Load()),
        slice=ast.Name(id=lib_name, ctx=ast.Load()),
        ctx=ast.Load()
    )
    
    # Build the append method call: .append(path_name)
    call = ast.Call(
        func=ast.Attribute(
            value=subscript,
            attr='append',
            ctx=ast.Load()
        ),
        args=[ast.Name(id=path_name, ctx=ast.Load())],
        keywords=[]
    )
    
    # Convert AST to code
    return ast.Expr(call)



def generate_self_dot_func(attr, args):
    dot_call = ast.Call(
        func=ast.Attribute(
            value=ast.Name(id='self', ctx=ast.Load()),  # Represents "self"
            attr=attr,                         # Represents ".add_source"
            ctx=ast.Load()
        ),
        args=args,      # Represents the argument "'example.txt'"
        keywords=[]                                    # No keyword arguments
    )
    return ast.Expr(dot_call)


def generate_initialize_method(class_name:str , dep_df: pd.DataFrame):
    method_name = "initialize"

    # generate instructions to load HDL sources
    add_source_instructions = []
    append_to_library_dict_instructions = []
    for i in range(len(dep_df)):
        lib, fpath = dep_df.loc[:, ["lib", "scilab_fpath"]].iloc[i]
        add_source_expr = generate_self_dot_func(attr='add_source', args=[ast.Constant(value=fpath)])
        append_to_lib_expr= generate_library_dict_append(class_name, lib, f"'{fpath}'")
        
        add_source_instructions.append(add_source_expr)
        append_to_library_dict_instructions.append(append_to_lib_expr)

    

    method_body = [
        ast.Expr(ast.Constant(value=f"{method_name.replace('_', ' ').title()} method")),
        generate_self_dot_func(attr='create_hdl_dir', args=[]),  # create hdl wrappers dir
        *add_source_instructions,
        *append_to_library_dict_instructions
    ]

    # generate method body
    method = ast.FunctionDef(
        name=method_name,
        args=ast.arguments(
            posonlyargs=[],
            args=[ast.arg(arg='self')],
            kwonlyargs=[],
            kw_defaults=[],
            defaults=[]
        ),
        body=method_body,
        decorator_list=[],
        lineno=1,  # Add default lineno
        end_lineno=1  # Add end_lineno for Python 3.9+
    )
    return method

In [14]:
"""
assemble the final DSP module
"""

def generate_dsp_block(dep_df, jc):
    name_idx = jc['parameters']['keys'].index('name')
    class_name = jc['parameters']['values'][name_idx]
    
    initialize_method = generate_initialize_method(class_name, dep_df)

    class_methods = {
        "initialize": initialize_method,
    }    
    
    # Generate the final module
    generated_code = generate_class(
        class_methods=class_methods,
        class_name=class_name
    )
    
    print(generated_code)
    return generated_code

dep_df = get_file_dependencies(target_fpath_test)
dsp_block_code = generate_dsp_block(dep_df, jc)

from collections import defaultdict
wideband_fft_top = defaultdict(list)

class wideband_fft_top(DSPBlock):

    def initialize(self):
        """Initialize method"""
        self.create_hdl_dir()
        self.add_source('casper_dspdevel/common_pkg/fixed_float_types_c.vhd')
        self.add_source('casper_dspdevel/common_pkg/fixed_pkg_c.vhd')
        self.add_source('casper_dspdevel/common_pkg/common_pkg.vhd')
        self.add_source('casper_dspdevel/common_components/common_pipeline.vhd')
        self.add_source('casper_dspdevel/casper_adder/common_add_sub.vhd')
        self.add_source('casper_dspdevel/common_components/common_async.vhd')
        self.add_source('casper_dspdevel/common_components/common_areset.vhd')
        self.add_source('casper_dspdevel/common_components/common_bit_delay.vhd')
        self.add_source('casper_dspdevel/common_components/common_pipeline_sl.vhd')
        self.add_source('casper_dspdevel/casper_multiplier/tech_mult_component.vhd')
        self.add_source('casper_dspdevel/casper_multiplier/tech_agilex_versal_cmult.vhd')
        self.add_source('casper_dspdevel/casper_multiplier/tech_complex_mult.vhd')
        self.add_source('casper_dspdevel/casper_multiplier/common_complex_mult.vhd')
        self.add_source('casper_dspdevel/casper_counter/common_counter.vhd')
        self.add_source('casper_dspdevel/common_components/common_delay.vhd')
        self.add_source('casper_dspdevel/casper_fifo/common_rl_decrease.vhd')
        self.add_source('casper_dspdevel/casper_fifo/common_fifo_rd.vhd')
        self.add_source('casper_dspdevel/casper_fifo/tech_fifo_component_pkg.vhd')
        self.add_source('casper_dspdevel/casper_fifo/tech_fifo_sc.vhd')
        self.add_source('casper_dspdevel/casper_fifo/common_fifo_sc.vhd')
        self.add_source('casper_dspdevel/casper_ram/common_ram_pkg.vhd')
        self.add_source('casper_dspdevel/casper_ram/tech_memory_component_pkg.vhd')
        self.add_source('casper_dspdevel/casper_ram/tech_memory_ram_crw_crw.vhd')
        self.add_source('casper_dspdevel/casper_ram/tech_memory_ram_cr_cw.vhd')
        self.add_source('casper_dspdevel/casper_ram/common_ram_crw_crw.vhd')
        self.add_source('casper_dspdevel/casper_ram/common_paged_ram_crw_crw.vhd')
        self.add_source('casper_dspdevel/casper_ram/common_paged_ram_rw_rw.vhd')
        self.add_source('casper_dspdevel/casper_ram/common_paged_ram_r_w.vhd')
        self.add_source('casper_dspdevel/casper_requantize/common_round.vhd')
        self.add_source('casper_dspdevel/casper_requantize/common_resize.vhd')
        self.add_source('casper_dspdevel/casper_requantize/common_requantize.vhd')
        self.add_source('casper_dspdevel/casper_ram/tech_memory_rom_r_r.vhd')
        self.add_source('casper_dspdevel/casper_ram/tech_memory_rom_r.vhd')
        self.add_source('casper_dspdevel/casper_ram/common_rom_r_r.vhd')
        self.add_source('casper_dspdevel/common_pkg/common_str_pkg.vhd')
        self.add_source('casper_dspdevel/casper_multiplexer/common_zip.vhd')
        self.add_source('casper_dspdevel/r2sdf_fft/twiddlesPkg.vhd')
        self.add_source('casper_dspdevel/r2sdf_fft/rTwoBF.vhd')
        self.add_source('casper_dspdevel/casper_requantize/r_shift_requantize.vhd')
        self.add_source('casper_dspdevel/r2sdf_fft/rTwoWMul.vhd')
        self.add_source('casper_dspdevel/casper_wb_fft/fft_r2_bf_par.vhd')
        self.add_source('casper_dspdevel/casper_wb_fft/fft_r2_par.vhd')
        self.add_source('casper_dspdevel/r2sdf_fft/rTwoBFStage.vhd')
        self.add_source('casper_dspdevel/r2sdf_fft/rTwoWeights.vhd')
        self.add_source('casper_dspdevel/r2sdf_fft/rTwoSDFStage.vhd')
        self.add_source('casper_dspdevel/casper_wb_fft/fft_sepa.vhd')
        self.add_source('casper_dspdevel/casper_wb_fft/fft_reorder_sepa_pipe.vhd')
        self.add_source('casper_dspdevel/casper_wb_fft/fft_r2_pipe.vhd')
        self.add_source('casper_dspdevel/casper_wb_fft/fft_sepa_wide.vhd')
        self.add_source('casper_d

### Visualize Python File AST

In [17]:
# In Jupyter notebook
from IPython.display import Image
import ast
from graphviz import Digraph

def extract_ast(code):
    """Extract abstract syntax tree from Python code"""
    tree = ast.parse(code)
    return tree

def visualize_ast(tree):
    """Create Graphviz visualization of AST"""
    dot = Digraph(comment='AST', node_attr={'shape': 'box', 'style': 'filled', 'fillcolor': '#f0f0f0'})
    
    def add_nodes_edges(node, parent=None):
        node_name = str(id(node))
        label = type(node).__name__
        
        # Add special handling for common node types
        if isinstance(node, ast.ClassDef):
            dot.node(node_name, f"Class: {node.name}", fillcolor='#e0f0e0')
        elif isinstance(node, ast.FunctionDef):
            dot.node(node_name, f"Function: {node.name}", fillcolor='#e0e0f0')
        elif isinstance(node, ast.Name):
            dot.node(node_name, f"Name: {node.id}", fillcolor='#f0e0f0')
        else:
            dot.node(node_name, label)
            
        if parent:
            dot.edge(parent, node_name)
            
        for child in ast.iter_child_nodes(node):
            add_nodes_edges(child, node_name)
            
    add_nodes_edges(tree)
    return dot

# Usage example

# with open('wbfft.py', 'r') as f:
#     code = f.read()

# 1. extract AST
ast_tree = extract_ast(dsp_block_code)
print(ast.dump(ast_tree, indent=4))  # Text representation

# 2. visualize AST
# dot = visualize_ast(ast_tree)
# dot.render('ast_graph', format='png', cleanup=True)  # Save as PNG
# dot  # Display in notebook if using Jupyter

# Image(dot.render(format='png'))



Module(
    body=[
        ImportFrom(
            module='collections',
            names=[
                alias(name='defaultdict')],
            level=0),
        Assign(
            targets=[
                Name(id='wideband_fft_top', ctx=Store())],
            value=Call(
                func=Name(id='defaultdict', ctx=Load()),
                args=[
                    Name(id='list', ctx=Load())],
                keywords=[])),
        ClassDef(
            name='wideband_fft_top',
            bases=[
                Name(id='DSPBlock', ctx=Load())],
            keywords=[],
            body=[
                FunctionDef(
                    name='initialize',
                    args=arguments(
                        posonlyargs=[],
                        args=[
                            arg(arg='self')],
                        kwonlyargs=[],
                        kw_defaults=[],
                        defaults=[]),
                    body=[
                        Expr(
                            value=Constant(value='Initialize method')),
                        Expr(
                            value=Call(
                                func=Attribute(
                                    value=Name(id='self', ctx=Load()),
                                    attr='create_hdl_dir',
                                    ctx=Load()),
                                args=[],
                                keywords=[])),
                        Expr(
                            value=Call(
                                func=Attribute(
                                    value=Name(id='self', ctx=Load()),
                                    attr='add_source',
                                    ctx=Load()),
                                args=[
                                    Constant(value='casper_dspdevel/common_pkg/fixed_float_types_c.vhd')],
                                keywords=[])),
                        Expr(
                            value=Call(
                                func=Attribute(
                                    value=Name(id='self', ctx=Load()),
                                    attr='add_source',
                                    ctx=Load()),
                                args=[
                                    Constant(value='casper_dspdevel/common_pkg/fixed_pkg_c.vhd')],
                                keywords=[])),
                        Expr(
                            value=Call(
                                func=Attribute(
                                    value=Name(id='self', ctx=Load()),
                                    attr='add_source',
                                    ctx=Load()),
                                args=[
                                    Constant(value='casper_dspdevel/common_pkg/common_pkg.vhd')],
                                keywords=[])),
                        Expr(
                            value=Call(
                                func=Attribute(
                                    value=Name(id='self', ctx=Load()),
                                    attr='add_source',
                                    ctx=Load()),
                                args=[
                                    Constant(value='casper_dspdevel/common_components/common_pipeline.vhd')],
                                keywords=[])),
                        Expr(
                            value=Call(
                                func=Attribute(
                                    value=Name(id='self', ctx=Load()),
                                    attr='add_source',
                                    ctx=Load()),
                                args=[
                                    Constant(value='casper_dspdevel/casper_adder/common_add_sub.vhd')],
                                keywords=[])),
                        Expr(
                            value=Call(
                                func=Attribute(
       